In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

from error_funcs import VelocityGridErrorCalculator
from utils import generate_time_intervals_seconds

In [ ]:
### STEP 0 -- Define variables
download_from_inception = False # if you already have the trajectory data in a csv somewhere, set this to false
calc_ground_truth_npy = True # if you already have the ground truth numpy file, set this to false
ground_truth_npy_path = "../reference_data/wb_velocity_1121_6_to_7_10s_400m-oct12.npy"
# ground_truth_npy_path = "../reference_data/wb_velocity_1121_6_to_7_10s_400m.npy"

duration_hrs = 1
# duration_hrs = 4

# date_hyphenated_list = ["11-21", "11-22", "11-23", "11-24", "11-25", "11-28", "11-29", "11-30", "12-01", "12-02"]
# date_hyphenated_list = ["11-30", "12-01", "12-02"]
date_hyphenated_list = ["11-21"]

for date_hyphenated in date_hyphenated_list:

    # for start_hour in [6, 7, 8, 9]:
    for start_hour in [6]:

        # date_hyphenated = "11-30" # date of interest in MM-DD format
        date_str = date_hyphenated.replace("-", "")
        # start_hour = 6
        # end_hour = start_hour + duration_hrs
        end_hour = start_hour + duration_hrs
        duration = duration_hrs * 60 # in minutes

        begin_time_in_seconds = start_hour * 60 * 60
        end_time_in_seconds = end_hour * 60 * 60
        # begin_time_in_seconds = 0
        # end_time_in_seconds = 7230

        inception_json_path = f"../i24_data/{date_hyphenated}.json"
        inception_csv_path = f"../i24_data/csvs/{date_str}_westbound_{start_hour}-{end_hour}.csv"

        min_time = pd.to_datetime(f"2022-{date_hyphenated} {start_hour + 6}:00:00.400000095") # The +6 is to convert from central time to utc time

        ### STEP 1 -- Load ground truth data from inception or inception pre-processed csv
        if download_from_inception:
            ### OPTION A: Get ground truth numpy array from inception trajectory -- disregard if you already possess it ###
            # if you want to adjust other settings such as space of interest, lane direction of interest, etc., look in the load_trajectories function itEC for now
            # inception_file_path = "../i24_data/11-29.json"

            trajectory_timeframe = pd.Timedelta(minutes=duration) # set based on however much time you want to load, starting from min_time
            inception_data_df = VelocityGridErrorCalculator.load_trajectories(inception_file_path=inception_json_path,
                                                trajectory_timeframe=trajectory_timeframe,
                                                min_time=min_time)

            # # it's a good idea to save this df to csv so you don't have to read the inception data every time for future runs
            # inception_data_df.to_csv("../i24_data/1129_eastbound_7_to_830.csv")
            inception_data_df.to_csv(inception_csv_path)
        else:
            # if you already have trajectory data in a csv somewhere
            inception_data_df = pd.read_csv(inception_csv_path)
            inception_data_df["timestamp"] = pd.to_datetime(inception_data_df["timestamp"])

        ## STEP 2 -- Initialize EC
        # When using your own data, change these paths accordingly. For some reason relative paths don't work here for me??
        # fcd_file_path = "../sim_files/fcd_output_mape120.xml"
        # fcd_file_path = "../cal_methods/bilevel/outputs/fcd_output.xml"
        fcd_file_path = "../cal_methods/bilevel/outputs/fcd_output.xml"
        # net_file_path = "../sim_files/sumo_test.net.xml"
        # net_file_path = "../cal_methods/bilevel/support_files/1130/opt1/i24_12-15_modified.net.xml"
        net_file_path = "../sim_files/sumo_test.net.xml"
        # mm_latlon_mapping_path = "../build_data/mile_marker_layer.csv"
        mm_latlon_mapping_path = "../build_data/mile_marker_layer.csv"

        # comment this out if you derived the gt numpy file from cells [2] and [3] above
        # ground_truth_npy_file = np.load("../reference_data/wb_velocity_7_to_8_10s_400m_flipped.npy")
        # ground_truth_npy_file = np.load("../reference_data/wb_velocity_7_to_8_10s_400m_flipped.npy")
        # print("ground truth npy shape:", ground_truth_npy_file.shape)

        # Generate time stamps for the period of interest.
        year_str = "2022"
        interval = 10 #seconds
        time_stamps = generate_time_intervals_seconds(year_str, date_str, begin_time_in_seconds, end_time_in_seconds, interval)

        # Initialize the VelocityGridErrorCalculator with the FCD data and other necessary files.
        EC = VelocityGridErrorCalculator(time_stamps, fcd_file_path, net_file_path, mm_latlon_mapping_path)


        ### STEP 3 -- Use EC to calculate ground truth numpy array if you don't already have it
        if calc_ground_truth_npy:
            # convert your df derived from the inception data file to a numpy array for plotting / error calculating
            gt_flow_npy, gt_density_npy, gt_velocity_npy, count_matrix = EC.get_ground_truth_npy(df = inception_data_df)

            gt_velocity_npy =  np.flip(gt_velocity_npy, axis=1)
            gt_velocity_npy = gt_velocity_npy[:,:]

            # change based on your metric of interest
            ground_truth_npy_file = gt_velocity_npy
            np.save(ground_truth_npy_path, ground_truth_npy_file)
            print("ground truth npy shape:", ground_truth_npy_file.shape)
            print("saved ground truth npy to:", ground_truth_npy_path)
        else:
            # load ground truth numpy
            ground_truth_npy_file = np.load(ground_truth_npy_path)

        print("ground truth npy shape:", ground_truth_npy_file.shape)

        ### STEP 4 -- Plot
        # plot the ground truth
        vmax = np.max(ground_truth_npy_file)
        plt.figure(figsize=(10, 6))
        plt.imshow(ground_truth_npy_file.T, aspect='auto', origin='lower', cmap='RdYlGn', interpolation="none", vmin = 0, vmax = 120)
        # Label axes
        plt.xlabel('Time Step')
        plt.ylabel('Segment Index')
        plt.title(f'Time-Space Diagram of Velocities - Ground Truth\n{date_str} Westbound {start_hour}:00 to {end_hour}:00')
        plt.colorbar(label='Average Velocity (km/hr)')
        plt.tight_layout()
        plt.savefig(f"../i24_data/figs/gt_velocity_{date_str}_westbound_{start_hour}-{end_hour}.png", dpi=300)
        plt.show()
        plt.close()

#### Now extract sim-produced data and compare

In [ ]:
# Compute the velocity grid from the FCD data.
eb_npy, eb_df, wb_npy, wb_df = EC.compute_velocity_grid_from_fcd(return_df = True)
# predicted_velocities_npy = predicted_velocities_npy[:,:-2]  # Remove the last column which corresponds to the "end" of the last spatial bin.
print(eb_npy.shape)
print(wb_npy.shape)
# glimpse at the velocity grid
eb_df.head()
# df.tail()
# predicted_velocities_npy.shape



In [ ]:
wb_npy.shape

In [ ]:
EC.mape(ground_truth_npy_file[:180, :], wb_npy[:180, :])

In [ ]:
# np.save("wb_velocity__7_to_8_10s_400m.npy", ground_truth_npy_file)

In [ ]:
# plot the ground truth
vmax = np.max(ground_truth_npy_file)
plt.figure(figsize=(10, 6))
plt.imshow(ground_truth_npy_file[:180, :].T, aspect='auto', origin='lower', cmap='RdYlGn', interpolation="none", vmin = 0, vmax = 120)
# Label axes
plt.xlabel('Time Step')
plt.ylabel('Segment Index')
plt.title('Time-Space Diagram of Velocities - Ground Truth')
plt.colorbar(label='Average Velocity (km/hr)')
plt.tight_layout()

# plot the FCD-derived velocity
plt.figure(figsize=(10, 6))
plt.imshow(eb_npy[:180, :].T, aspect='auto', origin='lower', cmap='RdYlGn', interpolation="none", vmin = 0, vmax=120)
# Label axes
plt.xlabel('Time Step')
plt.ylabel('Segment Index')
plt.title('Time-Space Diagram of Predictions')
plt.colorbar(label='Velocity (km/hr)')
plt.tight_layout()

plt.figure(figsize=(10, 6))
plt.imshow(wb_npy[:180, :].T, aspect='auto', origin='lower', cmap='RdYlGn', interpolation="none", vmin = 0, vmax=120)
# Label axes
plt.xlabel('Time Step')
plt.ylabel('Segment Index')
plt.title('Time-Space Diagram of Predictions')
plt.colorbar(label='Velocity (km/hr)')
plt.tight_layout()

In [ ]:
# file_path = "../sim_files/scen3_unnoised_1130_0420-0480/0/fcd_output.xml"

# with open(file_path, "r") as f:
#     for i in range(50):
#         line = f.readline()
#         if not line:
#             break  # stop if file is shorter than 20 lines
#         print(line.strip())

In [ ]:
# Set consistent vmax across both plots so colors are comparable
vmax = max(np.max(ground_truth_npy_file), np.max(predicted_velocities_npy))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# --- Ground Truth ---
im0 = axes[0].imshow(
    ground_truth_npy_file.T,
    aspect='auto',
    origin='lower',
    cmap='RdYlGn',
    interpolation="none",
    vmin=0,
    vmax=vmax
)
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Segment Index')
axes[0].set_title('Velocity Ground Truth')

# --- Predicted ---
im1 = axes[1].imshow(
    predicted_velocities_npy.T,
    aspect='auto',
    origin='lower',
    cmap='RdYlGn',
    interpolation="none",
    vmin=0,
    vmax=vmax
)
axes[1].set_xlabel('Time Step')
axes[1].set_title('Predicted Velocity')

# Add a single colorbar for both plots
fig.colorbar(im1, ax=axes, orientation='vertical', label='Velocity (km/hr)')

plt.show()

In [ ]:
# --- Load FCD file ---
fcd_xml_file_path = "../cal_methods/sumo_baseline/outputs/fcd_output.xml"
df = EC.fcd_xml_to_df(fcd_xml_file_path)
print(f"FCD DataFrame shape (before filtering): {df.shape}")

In [ ]:
df.head()
df.tail()


In [ ]:
 # Filter to only WB data
df['edge_id'] = df['lane'].apply(lambda x: x.split('_')[0])
valid_edge_ids = EC.wb_route_edge_ids

df['direction'] = '' 
# df['direction'] = df[df['edge_id'].isin(valid_edge_ids)]

df['direction'] = np.where(df['edge_id'].isin(EC.wb_route_edge_ids), 'wb',
                   np.where(df['edge_id'].isin(EC.eb_route_edge_ids), 'eb', 'neither'))

# df = df[df['edge_id'].isin(valid_edge_ids)
print(f"FCD DataFrame shape (after WB filter): {df.shape}")

In [ ]:
# --- Build interpolator for mile marker mapping ---
from scipy.interpolate import LinearNDInterpolator


lonlat_points = EC.mm_latlon_df[['X_WGS84', 'Y_WGS84']].values
mm_values = EC.mm_latlon_df['MM'].values
interp_func = LinearNDInterpolator(lonlat_points, mm_values)

# --- Map SUMO (x,y) → lon/lat → mile marker ---
lonlat = np.array([EC.net.convertXY2LonLat(x, y) for x, y in zip(df['x'], df['y'])])
df['mm'] = interp_func(lonlat)
# df = df.dropna(subset=['mm'])
print(f"FCD DataFrame shape (after mm mapping): {df.shape}")

df.head()
df.tail(10)

        

In [ ]:
nan_counts = df.groupby("edge_id")["mm"].apply(lambda x: x.isna().sum())
print(nan_counts)

nan_counts = df.groupby("edge_id")["mm"].apply(lambda x: x.isna().sum())

plt.figure(figsize=(10,6))
nan_counts.plot(kind="bar")


In [ ]:
nan_counts = df.groupby("direction")["mm"].apply(lambda x: x.isna().sum())
print(nan_counts)

nan_counts = df.groupby("direction")["mm"].apply(lambda x: x.isna().sum())

plt.figure(figsize=(10,6))
nan_counts.plot(kind="bar")

df.shape

In [ ]:
plt.figure(figsize=(8,6))

for direction, group in df.groupby("direction"):
    plt.hist(group["mm"].dropna(), bins=30, alpha=0.5, label=direction)

plt.title("Histogram of mm values by direction")
plt.xlabel("mm value")
plt.ylabel("Count")
plt.legend(title="Direction")
plt.show()

In [ ]:
df[df['direction'] == 'wb']

In [ ]:
df[(df['direction'] == 'wb') & (df['mm'].isna())]

In [ ]:
# --- Filter to road segment ---
x_min_miles = 58.8
x_max_miles = 62.8
df = df[(df['mm'] >= x_min_miles ) & (df['mm'] <= x_max_miles)]

if df.empty:
    raise ValueError("No FCD data in the specified segment and time range.")

print(max(df['mm']))
print(min(df['mm']))

new_df['mm'].unique()



In [ ]:
# --- Setup time bins based on micro_obj_function_timestamps ---
time_bins = np.array([VelocityGridErrorCalculator.to_seconds_since_midnight(ts) for ts in EC.micro_obj_fn_timestamps],dtype=float)
print(time_bins)

n_expected_bins = len(time_bins)
print(f"Using {n_expected_bins} time bins from micro_obj_function_timestamps")



In [ ]:
# t_min = df["time"].min()
# time_interval_sec = time_interval.total_seconds()

# if expected_duration is not None:
#     t_max = t_min + expected_duration
# else:
#     t_max = df["time"].max()

# n_expected_bins = int(np.ceil((t_max - t_min) / time_interval_sec))
# print(f"t_min = {t_min}, t_max = {t_max}, expected #bins = {n_expected_bins}")

# time_bins = np.arange(t_min, t_min + n_expected_bins * time_interval_sec + 1, time_interval_sec)

# --- Setup space bins ---
space_interval = 400
x_min = x_min_miles * 1609.34
x_max = x_max_miles * 1609.34
space_bins = np.arange(x_min, x_max + space_interval, space_interval)
print(space_bins)

df.head()


In [ ]:
# --- Assign bins ---
df["time_bin"] = pd.cut(df["time"], bins=time_bins, labels=False, include_lowest=True)
df["space_bin"] = pd.cut(df["mm"] * 1609.34, bins=space_bins, labels=False, include_lowest=True)

df.head()

In [ ]:

# test_2_df = df[(df['direction'] == 'wb') & (df['space_bin'] > 7)]
test_2_df = df[(df['direction'] == 'wb')]

plt.figure(figsize=(8,6))

for direction, group in test_2_df.groupby("direction"):
    plt.hist(group["mm"].dropna(), bins=30, alpha=0.5, label=direction)

plt.title("Histogram of mm values by direction")
plt.xlabel("mm value")
plt.ylabel("Count")
plt.legend(title="Direction")
plt.show()




In [ ]:
wb_df =  df[df['direction'] == 'wb']

In [ ]:
# --- Pivot to grid ---
pivot_table = wb_df.pivot_table(
    index="time_bin",
    columns="space_bin",
    values="speed",
    aggfunc="mean"
)

pivot_table.head()


In [ ]:
# # Reindex to full grid (pads with NaN if missing)
# pivot_table_new = pivot_table.reindex(
#     index=range(n_expected_bins),
#     columns=range(len(space_bins) - 1)
# )
# pivot_table_new.head()

In [ ]:
# Convert m/s → km/h directly in the DataFrame
pivot_table = pivot_table * 3.6

velocity_grid = np.flip(pivot_table.to_numpy(), axis = 1)  # Flip so first last segments are on left (for visualization and error calc consistency)

# if return_df:
#     return velocity_grid, time_bins, space_bins, pivot_table
# else:
#     return velocity_grid, time_bins, space_bins

In [ ]:
plt.figure(figsize=(10, 6))
plt.imshow(velocity_grid.T, aspect='auto', origin='lower', cmap='RdYlGn', interpolation="none", vmin = 0, vmax=100)
# Label axes
plt.xlabel('Time Step')
plt.ylabel('Segment Index')
plt.title('Time-Space Diagram of Predictions')
plt.colorbar(label='Velocity (km/hr)')
plt.tight_layout()

In [ ]:
df = inception_data_df.copy()

t_min, t_max = df["timestamp"].min(), df["timestamp"].max()
x_min, x_max = df["x_position"].min(), df["x_position"].max()
print("xmax", x_max)
print("x_min", x_min)
# Ensure valid ranges
if x_min == x_max:
    raise ValueError("x_min and x_max are identical, meaning no variation in x_position.")

df["x_position_mm"] = df["x_position"] / 5280 / 0.3048
df['eb_transfer_pos'] = 62.8 - df['x_position_mm'] + 58.8



df["eb_transfer_pos_m"] = df["eb_transfer_pos"] * 5280 * 0.3048
x_min, x_max = df["eb_transfer_pos_m"].min(), df["eb_transfer_pos_m"].max()
print("xmax", x_max)
print("x_min", x_min)

df.head()

df['x_position_old'] = df['x_position']
df['x_position'] = df['eb_transfer_pos_m']

eb_conversion_df = df
eb_conversion_df.head()

In [ ]:
# Create time and space bins
time_bins = pd.date_range(start=t_min, end=t_max, freq=time_interval)
print(time_bins)
space_bins = np.arange(x_min, x_max, space_interval)
print(space_bins)

if len(space_bins) < 2:
    raise ValueError("space_bins array is empty or too small, adjust space_interval.")

# labels = [58.8, 59.0666666667, 59.3333333333, 59.6, 
# 59.8666666667, 60.1333333333, 60.4, 60.6666666667, 
# 60.9333333333, 61.2, 61.4666666667, 61.7333333333, 
# 62.0, 62.2666666667, 62.5333333333, 62.8]

# Assign bin indices using `pd.cut()`
df["time_bin"] = pd.cut(df["timestamp"], bins=time_bins, labels=False, include_lowest=True)
df["space_bin"] = pd.cut(df["eb_transfer_pos_m"], bins=space_bins, labels=labels, include_lowest=True)

# Remove NaNs (out-of-range values)
df = df.dropna(subset=["time_bin", "space_bin"]).astype({"time_bin": int, "space_bin": int})

# Compute flow and density using `groupby()`
flow_matrix = np.zeros((len(time_bins) - 1, len(space_bins) - 1))
density_matrix = np.zeros_like(flow_matrix)
# lane_matrix = np.zeros_like(flow_matrix)